#Instalasi Library
Install semua dependency yang dibutuhkan.

In [1]:
# Install semua library yang dibutuhkan
!pip install -q langchain langchain-community chromadb sentence-transformers
!pip install -q -U google-generativeai
!pip install -q pyngrok streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

#Mount Google Drive
Mount Google Drive untuk mengakses dataset.

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


#Load Dataset
Membaca dataset hasil preprocessing dari Google Drive.

In [3]:
import pandas as pd

df = pd.read_csv(
    '/content/drive/MyDrive/Dataset/data_hasil_preprocessing (3).csv',
    sep=',',
    encoding='latin-1'
)
df.columns = df.columns.str.strip()

print(f'Dataset shape: {df.shape}')
print(f'Kolom: {df.columns.tolist()}')
df.head()


Dataset shape: (1060, 10)
Kolom: ['Segmentasi', 'Status', 'New/ Existing', 'Product', 'offer status', 'B/W (Mbps)', 'OTC', 'MRC', 'GRAND TOTAL', 'Cluster']


,Segmentasi,Status,New/ Existing,Product,offer status,B/W (Mbps),OTC,MRC,GRAND TOTAL,Cluster
0,EDUCATION,Greenfield,NEW,SMARTLINK,LOST,0.0050,0.027273,0.015294,0.026052,4
1,RETAIL CONSUMER GOODS,in Building,NEW,SMARTLINK,ACTIVE,0.0030,0.000000,0.035294,0.028056,5
2,EDUCATION,Greenfield,NEW,INFYLINK MIX,LOST,0.0050,0.018182,0.015294,0.021042,4
3,FINANCIAL & BANKING,Greenfield,NEW,SMARTLINK,DISSENGGAGED,0.0005,0.000000,0.017647,0.013026,5
4,EDUCATION,Greenfield,NEW,SMARTLINK,LOST,0.0100,0.018182,0.027059,0.031062,4


#📝 Generate Dokumen Cluster
Mengubah setiap cluster menjadi teks deskriptif untuk RAG knowledge base.

In [4]:
# List untuk menyimpan dokumen cluster
cluster_docs = []

for cluster_id in sorted(df['Cluster'].unique()):
    cluster_data = df[df['Cluster'] == cluster_id]

    # Informasi dominan
    dominant_segment = cluster_data['Segmentasi'].mode()[0]
    dominant_status  = cluster_data['Status'].mode()[0]
    dominant_product = cluster_data['Product'].mode()[0]
    dominant_offer   = cluster_data['offer status'].mode()[0]

    # Statistik numerik
    avg_bw    = cluster_data['B/W (Mbps)'].mean()
    avg_otc   = cluster_data['OTC'].mean()
    avg_mrc   = cluster_data['MRC'].mean()
    avg_total = cluster_data['GRAND TOTAL'].mean()
    total_customer = len(cluster_data)

    text = f"""
    Cluster {cluster_id} terdiri dari {total_customer} data pelanggan.
    Cluster ini didominasi oleh segmentasi {dominant_segment} dengan status {dominant_status}.
    Produk yang paling banyak digunakan adalah {dominant_product}.
    Mayoritas pelanggan memiliki offer status {dominant_offer}.
    Nilai rata-rata bandwidth sebesar {avg_bw:.4f} Mbps,
    rata-rata OTC sebesar {avg_otc:.4f},
    rata-rata MRC sebesar {avg_mrc:.4f},
    dan rata-rata GRAND TOTAL sebesar {avg_total:.4f}.
    Cluster ini merepresentasikan pola pelanggan berdasarkan
    aktivitas sales, produk layanan, dan potensi revenue.
    """
    cluster_docs.append(text)

# Preview hasil
for i, doc in enumerate(cluster_docs):
    print(f'\n============ CLUSTER {i} ============')
    print(doc)



============ CLUSTER 0 ============

    Cluster 0 terdiri dari 362 data pelanggan.
    Cluster ini didominasi oleh segmentasi MANUFAKTUR dengan status in Building.
    Produk yang paling banyak digunakan adalah SMARTLINK.
    Mayoritas pelanggan memiliki offer status LOST .
    Nilai rata-rata bandwidth sebesar 0.0073 Mbps,
    rata-rata OTC sebesar 0.0409,
    rata-rata MRC sebesar 0.0295,
    dan rata-rata GRAND TOTAL sebesar 0.0457.
    Cluster ini merepresentasikan pola pelanggan berdasarkan
    aktivitas sales, produk layanan, dan potensi revenue.
    

============ CLUSTER 1 ============

    Cluster 1 terdiri dari 88 data pelanggan.
    Cluster ini didominasi oleh segmentasi EDUCATION dengan status Greenfield.
    Produk yang paling banyak digunakan adalah INFYLINK MIX.
    Mayoritas pelanggan memiliki offer status PROPOSAL.
    Nilai rata-rata bandwidth sebesar 0.0129 Mbps,
    rata-rata OTC sebesar 0.0530,
    rata-rata MRC sebesar 0.0902,
    dan rata-rata GRAND TOTAL sebes

#Membuat Embedding
Mengkonversi teks cluster menjadi vector embedding menggunakan Sentence Transformers.

In [5]:
from sentence_transformers import SentenceTransformer

# Load model embedding
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
embeddings = embedding_model.encode(cluster_docs)

# Verifikasi hasil
print(f'Tipe  : {type(embeddings)}')
print(f'Jumlah: {len(embeddings)}')
print(f'Dimensi vektor: {len(embeddings[0])}')
print(f'Contoh 10 nilai pertama:\n{embeddings[0][:10]}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Tipe  : <class 'numpy.ndarray'>
Jumlah: 6
Dimensi vektor: 384
Contoh 10 nilai pertama:
[ 0.01554127 -0.01511977 -0.04516343 -0.0440736  -0.07651323 -0.0386756
  0.00049188  0.04298618 -0.05101479 -0.00559138]


#Membuat Vector Database (ChromaDB)
Menyimpan embedding ke ChromaDB untuk semantic search.

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Inisialisasi embedding function
embedding_function = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# Buat vector database dari dokumen cluster
# Catatan: vectordb.persist() sudah deprecated di ChromaDB versi baru,
# data otomatis tersimpan jika persist_directory diset.
vectordb = Chroma.from_texts(
    texts=cluster_docs,
    embedding=embedding_function,
    persist_directory='cluster_db'
)

print(f'Vector DB berhasil dibuat dengan {vectordb._collection.count()} dokumen.')


/tmp/ipykernel_6916/418844713.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector DB berhasil dibuat dengan 6 dokumen.


#Retrieval / Semantic Search
Mencari dokumen cluster yang relevan berdasarkan query semantik.

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load vector DB yang sudah dibuat
embedding_function = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

vectordb = Chroma(
    persist_directory='cluster_db',
    embedding_function=embedding_function
)

retriever = vectordb.as_retriever(search_kwargs={'k': 3})
print('Retriever berhasil diinisialisasi.')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retriever berhasil diinisialisasi.


/tmp/ipykernel_6916/4072352161.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


In [8]:
# Contoh query retrieval
test_queries = [
    'Cluster dengan pelanggan aktif tertinggi',
    'Cluster dengan revenue terbesar',
    'Cluster paling berisiko churn',
    'Cluster yang cocok untuk upselling',
]

for query in test_queries:
    print(f'\n{'='*50}')
    print(f'Query: {query}')
    print('='*50)
    docs = retriever.invoke(query)
    for i, doc in enumerate(docs):
        print(f'\n--- Dokumen {i+1} ---')
        print(doc.page_content[:300], '...')



Query: Cluster dengan pelanggan aktif tertinggi

--- Dokumen 1 ---

    Cluster 5 terdiri dari 288 data pelanggan.
    Cluster ini didominasi oleh segmentasi ISP & ICT dengan status in Building.
    Produk yang paling banyak digunakan adalah SMARTLINK.
    Mayoritas pelanggan memiliki offer status PROPOSAL.
    Nilai rata-rata bandwidth sebesar 0.0084 Mbps,
    rat ...

--- Dokumen 2 ---

    Cluster 2 terdiri dari 99 data pelanggan.
    Cluster ini didominasi oleh segmentasi FINANCIAL & BANKING dengan status in Building.
    Produk yang paling banyak digunakan adalah INFINYLINK.
    Mayoritas pelanggan memiliki offer status PROPOSAL.
    Nilai rata-rata bandwidth sebesar 0.0065 Mbp ...

--- Dokumen 3 ---

    Cluster 4 terdiri dari 143 data pelanggan.
    Cluster ini didominasi oleh segmentasi EDUCATION dengan status Greenfield.
    Produk yang paling banyak digunakan adalah INFINYLINK.
    Mayoritas pelanggan memiliki offer status LOST .
    Nilai rata-rata bandwidth sebesar 0.0084 

# 🔑 Konfigurasi API Key Gemini

> **⚠️ PENTING:** Jangan pernah hardcode API key di dalam kode.
> Gunakan `google.colab.userdata` (cara paling aman di Colab) atau environment variable.

**Cara set API Key di Colab:**
1. Klik ikon 🔑 **Secrets** di sidebar kiri Colab
2. Tambahkan secret dengan nama `GEMINI_API_KEY`
3. Paste API key kamu sebagai nilainya
4. Aktifkan toggle "Notebook access"


In [9]:
import google.generativeai as genai
from google.colab import userdata

# ============================================================
# CARA AMAN: Ambil API key dari Colab Secrets
# ============================================================
# Pastikan kamu sudah set secret 'GEMINI_API_KEY' di sidebar Colab
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Konfigurasi Gemini
genai.configure(api_key=GEMINI_API_KEY)
print('API Key berhasil dikonfigurasi.')

# Verifikasi: list model yang tersedia
print('\nModel yang tersedia:')
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(f'  - {m.name}')


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


API Key berhasil dikonfigurasi.

Model yang tersedia:
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.5-flash
  - models/lyria-3-clip-preview
  - models/lyria-3-pro-preview
  - models/gemini-3.1-flash-tts-preview

# 🤖 Inisialisasi Model Gemini
Menggunakan `gemini-1.5-flash` sebagai model LLM untuk RAG pipeline.

> Model `gemini-3.1-flash-lite-preview` tidak valid. Model yang benar: `gemini-1.5-flash` atau `gemini-2.0-flash`.

In [10]:
# Gunakan model yang valid
# Pilihan: 'gemini-1.5-flash' (stabil) atau 'gemini-2.0-flash' (terbaru)
model = genai.GenerativeModel('models/gemini-2.5-flash')

# Test koneksi
response = model.generate_content('Halo, tolong konfirmasi kamu aktif.')
print('Response model:', response.text)


Response model: Halo! Ya, saya aktif. Bagaimana saya bisa membantu Anda?


#Full RAG Pipeline
Menggabungkan retrieval ChromaDB dengan generasi jawaban dari Gemini.

In [11]:
import google.generativeai as genai
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from google.colab import userdata

# ============================================================
# KONFIGURASI
# ============================================================
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel('models/gemini-2.5-flash')

# ============================================================
# LOAD VECTOR DB
# ============================================================
embedding_function = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

vectordb = Chroma(
    persist_directory='cluster_db',
    embedding_function=embedding_function
)

retriever = vectordb.as_retriever(search_kwargs={'k': 3})

# ============================================================
# FUNGSI RAG
# ============================================================
def rag_query(query: str) -> str:
    """Jalankan RAG pipeline: retrieval + generation."""
    # Retrieval
    docs = retriever.invoke(query)
    context = '\n'.join([doc.page_content for doc in docs])

    # Prompt
    prompt = f"""
    Kamu adalah AI Business Analyst Bali Fiber.
    Gunakan context berikut untuk menjawab pertanyaan user.

    Context:
    {context}

    Pertanyaan:
    {query}

    Berikan:
    1. Insight bisnis
    2. Analisis cluster
    3. Rekomendasi strategi
    """

    # Generate
    response = model.generate_content(prompt)
    return response.text

# ============================================================
# TEST
# ============================================================
query = 'Cluster mana yang memiliki risiko churn tertinggi?'
print(f'Query: {query}\n')
print(rag_query(query))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: Cluster mana yang memiliki risiko churn tertinggi?

Berdasarkan konteks yang diberikan, **Cluster 4** adalah cluster yang memiliki risiko churn tertinggi.

Berikut adalah analisis dan rekomendasi:

### 1. Insight Bisnis

Cluster 4, dengan 143 data pelanggan, menunjukkan pola yang sangat mengkhawatirkan karena **mayoritas pelanggan memiliki offer status LOST**. Ini mengindikasikan kegagalan signifikan dalam proses akuisisi atau retensi pelanggan. Meskipun segmennya didominasi oleh EDUCATION dengan status Greenfield (potensi pelanggan baru), tingginya status "LOST" menunjukkan Bali Fiber kehilangan banyak peluang di segmen ini. Selain itu, nilai rata-rata bandwidth dan potensi pendapatan (OTC, MRC, GRAND TOTAL) di cluster ini cenderung lebih rendah dibandingkan Cluster 1, yang juga memiliki segmen EDUCATION dan status Greenfield. Ini berarti tidak hanya ada masalah dalam mengonversi pelanggan, tetapi juga potensi nilai yang lebih rendah per pelanggan jika mereka berhasil dikonvers

#Streamlit App (app.py)
Buat file `app.py` untuk UI Streamlit.

> **Catatan:** API Key dibaca dari environment variable `GEMINI_API_KEY`, bukan hardcoded.

In [12]:
%%writefile app.py

import os
import streamlit as st
import google.generativeai as genai
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# ============================================
# KONFIGURASI HALAMAN
# ============================================
st.set_page_config(
    page_title='Bali Fiber AI Assistant',
    page_icon='🤖',
    layout='wide',
    initial_sidebar_state='collapsed'
)

# ============================================
# CUSTOM CSS
# ============================================
st.markdown("""
<style>
.stApp {
    background: linear-gradient(135deg, #000000 0%, #0A0A0A 40%, #111111 100%);
    color: white;
}
.block-container { padding-top: 2rem; padding-bottom: 2rem; }
.main-title {
    font-size: 42px; font-weight: 800;
    background: linear-gradient(90deg, #FF8C00, #FFA733, #FFD27A);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    margin-bottom: 5px; text-align: center; letter-spacing: 1px;
}
.subtitle { font-size: 18px; color: #B8C1EC; margin-bottom: 35px; text-align: center; }
.section-title { font-size: 28px; font-weight: 700; margin: 25px 0; color: #FFB347; text-align: center; }
.footer { text-align: center; color: #7F8DB0; margin-top: 50px; font-size: 14px; }
.result-box {
    background: rgba(20,20,20,0.85); backdrop-filter: blur(12px);
    border: 1px solid #FF8C00; box-shadow: 0 8px 25px rgba(0,0,0,0.4);
}
div.stButton > button {
    width: 100%; height: 70px; border-radius: 16px;
    border: 1px solid #FF8C00;
    background: linear-gradient(145deg, #121212, #1C1C1C);
    color: white; font-size: 16px; font-weight: 600;
    box-shadow: 0 4px 15px rgba(255,140,0,0.15); transition: 0.3s;
    margin-bottom: 15px;
}
div.stButton > button:hover {
    background: linear-gradient(145deg, #1F1F1F, #2C2C2C);
    border: 1px solid #FFA733;
    box-shadow: 0 6px 18px rgba(255,140,0,0.35);
    transform: translateY(-2px);
}
.stTextInput > div > div > input {
    background-color: #111111; color: white;
    border: 1px solid #FF8C00; border-radius: 14px;
    padding: 14px; font-size: 16px;
}
.streamlit-expanderHeader { border: 1px solid #FF8C00 !important; border-radius: 10px; }
@media (max-width: 768px) {
    .main-title { font-size: 30px; }
    .subtitle { font-size: 15px; }
    .section-title { font-size: 22px; }
    div.stButton > button { height: 65px; font-size: 14px; }
}
</style>
""", unsafe_allow_html=True)

# ============================================
# HEADER
# ============================================
st.markdown("""
<div class="main-title">Bali Fiber AI Assistant</div>
<div class="subtitle">
    Sistem AI berbasis RAG untuk analisis segmentasi pelanggan
    dan rekomendasi strategi bisnis Bali Fiber.
</div>
""", unsafe_allow_html=True)

# ============================================
# KONFIGURASI API KEY
# FIX: Baca dari environment variable, bukan hardcode
# ============================================
API_KEY = os.environ.get('GEMINI_API_KEY')

if not API_KEY:
    st.error('❌ GEMINI_API_KEY tidak ditemukan. Set environment variable sebelum menjalankan app.')
    st.stop()

genai.configure(api_key=API_KEY)

# ============================================
# MODEL GEMINI
# FIX: Nama model yang valid
# ============================================
model = genai.GenerativeModel('models/gemini-2.5-flash')

# ============================================
# LOAD EMBEDDING MODEL
# ============================================
@st.cache_resource
def load_embedding():
    return HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2'
    )

# ============================================
# LOAD VECTOR DATABASE
# ============================================
@st.cache_resource
def load_vectordb(_embedding_fn):
    return Chroma(
        persist_directory='cluster_db',
        embedding_function=_embedding_fn
    )

embedding_function = load_embedding()
vectordb = load_vectordb(embedding_function)
retriever = vectordb.as_retriever(search_kwargs={'k': 3})

# ============================================
# REKOMENDASI PERTANYAAN
# ============================================
st.markdown('<div class="section-title">💡 Rekomendasi Pertanyaan</div>', unsafe_allow_html=True)

questions = [
    ('📊 Cluster mana yang paling potensial?',      'Cluster mana yang paling potensial?'),
    ('⚠️ Cluster dengan risiko churn tertinggi?',   'Cluster dengan risiko churn tertinggi?'),
    ('👥 Cluster pelanggan aktif terbanyak?',        'Cluster mana yang memiliki pelanggan aktif terbanyak?'),
    ('📌 Apa karakteristik Cluster 0?',              'Apa karakteristik Cluster 0?'),
    ('🚀 Cluster prioritas untuk sales?',            'Cluster mana yang perlu diprioritaskan sales?'),
    ('📈 Cluster dengan performa terbaik?',          'Cluster dengan performa terbaik?'),
]

for i in range(0, len(questions), 2):
    col1, col2 = st.columns(2)
    with col1:
        label1, query1 = questions[i]
        if st.button(label1, key=f'btn_{i}'):
            st.session_state.query = query1
    with col2:
        if i + 1 < len(questions):
            label2, query2 = questions[i + 1]
            if st.button(label2, key=f'btn_{i+1}'):
                st.session_state.query = query2

# ============================================
# INPUT USER
# ============================================
query = st.text_input(
    '💬 Masukkan Pertanyaan:',
    value=st.session_state.get('query', ''),
    placeholder='Contoh: Cluster mana yang paling berisiko churn?'
)

# ============================================
# VALIDASI KATA KUNCI
# ============================================
allowed_keywords = [
    'cluster', 'pelanggan', 'sales', 'segmentasi',
    'churn', 'potensial', 'retensi', 'strategi',
    'aktif', 'bali fiber', 'performa'
]

# ============================================
# PROSES RAG
# ============================================
if query:
    if not any(kw in query.lower() for kw in allowed_keywords):
        st.warning('⚠️ Pertanyaan di luar konteks Business Insight Bali Fiber.')
        st.info('Silakan ajukan pertanyaan terkait cluster pelanggan, strategi bisnis, sales, atau segmentasi.')
    else:
        with st.spinner('🔍 AI sedang menganalisis data cluster...'):
            try:
                docs = retriever.invoke(query)
                context = '\n'.join([doc.page_content for doc in docs])
                prompt = f"""
                Kamu adalah AI Business Analyst Bali Fiber.
                Gunakan context berikut untuk menjawab pertanyaan user.
                Context:
                {context}
                Pertanyaan:
                {query}
                Berikan:
                1. Insight bisnis
                2. Analisis cluster
                3. Rekomendasi strategi
                """
                response = model.generate_content(prompt)
                hasil_ai = response.text
            except Exception as e:
                context = 'Data cluster sementara tidak tersedia.'
                st.warning(f'⚠️ Error: {e}')
                hasil_ai = ''

        if hasil_ai:
            st.markdown('## 📊 Hasil Analisis AI')
            st.write(hasil_ai)
            with st.expander('📁 Lihat Context Retrieval'):
                for i, doc in enumerate(docs):
                    st.markdown(f'### Cluster Retrieval {i+1}')
                    st.write(doc.page_content)

# ============================================
# FOOTER
# ============================================
st.markdown(
    '<div class="footer">Developed using K-Prototypes Clustering + RAG + Gemini LLM</div>',
    unsafe_allow_html=True
)


Writing app.py


#Jalankan Aplikasi Streamlit

> **⚠️ PENTING:** Ngrok authtoken juga jangan hardcode.
> Set sebagai secret `NGROK_TOKEN` di Colab Secrets.

In [13]:
from pyngrok import ngrok
from google.colab import userdata
import threading
import os

# ============================================================
# Set environment variable agar app.py bisa baca API key
# ============================================================
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

# ============================================================
# Autentikasi ngrok dari Colab Secrets
# ============================================================
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

# ============================================================
# Jalankan Streamlit di thread terpisah
# ============================================================
def run_streamlit():
    os.system('streamlit run app.py --server.port 8501 --server.enableCORS false')

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

# Buka tunnel ngrok
public_url = ngrok.connect(8501)
print('✅ Aplikasi berjalan di:', public_url)


✅ Aplikasi berjalan di: NgrokTunnel: "https://ef05-34-31-66-114.ngrok-free.app" -> "http://localhost:8501"



Jalankan cell ini untuk menghentikan tunnel ngrok.

In [14]:
#from pyngrok import ngrok
#ngrok.kill()
#print('Ngrok tunnel dihentikan.')
